In [12]:
import sys
from pathlib import Path

PROJECT_DIR = Path.home() / "entexBERT-2"
sys.path.insert(0, str(PROJECT_DIR / "src"))

RUN_DIR = PROJECT_DIR / "AS/1/CTCF/all_tissues/hap_pair_classification"
PATCHED_MODEL = PROJECT_DIR / "DNABERT-2-117M-attention"

CHECKPOINT_DIR = RUN_DIR / "output"
EXAMPLES_CSV = RUN_DIR / "analysis/test/representative_examples/representative_examples_all.csv"

print("Project dir:", PROJECT_DIR)
print("Patched model:", PATCHED_MODEL, PATCHED_MODEL.exists())
print("Checkpoint dir:", CHECKPOINT_DIR, CHECKPOINT_DIR.exists())
print("Examples CSV:", EXAMPLES_CSV, EXAMPLES_CSV.exists())

Project dir: /home/asm242/entexBERT-2
Patched model: /home/asm242/entexBERT-2/DNABERT-2-117M-attention True
Checkpoint dir: /home/asm242/entexBERT-2/AS/1/CTCF/all_tissues/hap_pair_classification/output True
Examples CSV: /home/asm242/entexBERT-2/AS/1/CTCF/all_tissues/hap_pair_classification/analysis/test/representative_examples/representative_examples_all.csv True


In [13]:
import json
import re

import torch
import pandas as pd
import transformers

from bertviz import head_view, model_view
from entexbert2.finetune_entexbert2 import entexBERT2ForSequencePrediction

In [14]:
def find_best_or_final_model_file(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)

    trainer_state_path = checkpoint_dir / "trainer_state.json"
    if trainer_state_path.exists():
        with open(trainer_state_path) as f:
            state = json.load(f)

        best_ckpt = state.get("best_model_checkpoint")
        if best_ckpt is not None:
            best_ckpt = Path(best_ckpt)
            for fname in ["model.safetensors", "pytorch_model.bin"]:
                candidate = best_ckpt / fname
                if candidate.exists():
                    return candidate

    for fname in ["model.safetensors", "pytorch_model.bin"]:
        candidate = checkpoint_dir / fname
        if candidate.exists():
            return candidate

    checkpoint_paths = []
    for p in checkpoint_dir.glob("checkpoint-*"):
        match = re.search(r"checkpoint-(\d+)$", str(p))
        if match:
            checkpoint_paths.append((int(match.group(1)), p))

    if checkpoint_paths:
        checkpoint_paths.sort()
        latest = checkpoint_paths[-1][1]
        for fname in ["model.safetensors", "pytorch_model.bin"]:
            candidate = latest / fname
            if candidate.exists():
                return candidate

    raise FileNotFoundError(f"Could not find model weights in {checkpoint_dir}")

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = transformers.AutoTokenizer.from_pretrained(
    str(PATCHED_MODEL),
    trust_remote_code=True,
    model_max_length=512,
    padding_side="right",
    use_fast=True,
)

model = entexBERT2ForSequencePrediction(
    model_name_or_path=str(PATCHED_MODEL),
    main_task="classification",
    main_num_labels=2,
    pooling_mode="cls",
    head_num_layers=1,
    head_hidden_size=-1,
    head_activation="gelu",
    head_dropout=0.1,
)

checkpoint = find_best_or_final_model_file(CHECKPOINT_DIR)
state = torch.load(checkpoint, map_location="cpu")

missing, unexpected = model.load_state_dict(state, strict=False)
print("Checkpoint:", checkpoint)
print("Missing:", len(missing))
print("Unexpected:", len(unexpected))

model.to(device)
model.eval()

Some weights of the model checkpoint at /home/asm242/entexBERT-2/DNABERT-2-117M-attention were not used when initializing BertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertModel were not initialized from the model checkpoint at /home/asm242/entexBERT-2/DNABERT-2-117M-attention and are newly initialized: ['bert.pooler.dense.we

Checkpoint: /home/asm242/entexBERT-2/AS/1/CTCF/all_tissues/hap_pair_classification/output/checkpoint-8600/pytorch_model.bin
Missing: 0
Unexpected: 0


entexBERT2ForSequencePrediction(
  (backbone): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(4096, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertUnpadAttention(
            (self): BertUnpadSelfAttention(
              (dropout): Dropout(p=0.0, inplace=False)
              (Wqkv): Linear(in_features=768, out_features=2304, bias=True)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (mlp): BertGatedLinearUnitMLP(
            (gated_layers): Linear(in_features=768, out_fea

In [16]:
examples = pd.read_csv(EXAMPLES_CSV)

category = "TP"   # try "TP", "TN", "FP", "FN"
rank = 1

sub = examples[examples["confusion_category"] == category].copy()

if "selection_rank_within_category" in sub.columns:
    row = sub[sub["selection_rank_within_category"] == rank].iloc[0]
else:
    row = sub.iloc[rank - 1]

seq1 = str(row["sequence1"])
seq2 = str(row["sequence2"])

row[["label", "pred_label", "prob_positive", "confusion_category"]]

label                        1
pred_label                   1
prob_positive         0.994207
confusion_category          TP
Name: 0, dtype: object

In [17]:
inputs = tokenizer(
    seq1,
    seq2,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=512,
)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

batch = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(
        **batch,
        output_attentions=True,
        output_hidden_states=True,
    )

probs = torch.softmax(outputs.logits, dim=-1)[0].detach().cpu()

print("logits:", outputs.logits.detach().cpu())
print("prob non-AS:", float(probs[0]))
print("prob AS:", float(probs[1]))
print("num tokens:", len(tokens))
print("num attention layers:", len(outputs.attentions))
print("attention[0] shape:", outputs.attentions[0].shape)

logits: tensor([[-2.5398,  2.6056]])
prob non-AS: 0.005792804528027773
prob AS: 0.9942071437835693
num tokens: 215
num attention layers: 12
attention[0] shape: torch.Size([1, 12, 215, 215])


In [18]:
sequence_ids = inputs.sequence_ids(0)

sentence_b_start = None
for i, sid in enumerate(sequence_ids):
    if sid == 1:
        sentence_b_start = i
        break

print("sentence_b_start:", sentence_b_start)

for i, (tok, sid) in enumerate(zip(tokens, sequence_ids)):
    print(i, sid, tok)

sentence_b_start: 108
0 None [CLS]
1 0 AA
2 0 CTGAA
3 0 TTACTT
4 0 TCTA
5 0 TCCAGTT
6 0 CCACTG
7 0 GTT
8 0 CGCA
9 0 GTGTTTTA
10 0 TCTT
11 0 TATATT
12 0 AAAA
13 0 CATATA
14 0 TAAA
15 0 TGATT
16 0 GATT
17 0 GATGAA
18 0 TTTTTTTT
19 0 CTTTTA
20 0 GACA
21 0 CTGCTG
22 0 CTGAA
23 0 TCCATG
24 0 GAGA
25 0 GAAAAA
26 0 GGA
27 0 TAAATT
28 0 TCCAGAA
29 0 CTATG
30 0 GTCC
31 0 CTGTG
32 0 CTC
33 0 CAGTT
34 0 CGCCAGG
35 0 CGG
36 0 GCGCGG
37 0 CGGA
38 0 GACGGA
39 0 GACC
40 0 GAGGAA
41 0 CGC
42 0 GGCTG
43 0 GGGC
44 0 CATG
45 0 CGG
46 0 CGCTA
47 0 CC
48 0 GCGC
49 0 GTGGTG
50 0 GC
51 0 GCTGTG
52 0 TCTG
53 0 GCC
54 0 TGAGG
55 0 CTTCTG
56 0 CTC
57 0 GCTCC
58 0 TTTA
59 0 CGCC
60 0 TTCA
61 0 GCCA
62 0 GCTCC
63 0 CCATG
64 0 TCCCC
65 0 GGAGGAA
66 0 GGA
67 0 GCGG
68 0 GCGGTG
69 0 GTGGC
70 0 GGGAA
71 0 GCTGCAGG
72 0 CC
73 0 GCAGTG
74 0 GCTT
75 0 CCTG
76 0 GCTG
77 0 GCGG
78 0 GAGG
79 0 CGGATG
80 0 CGGTG
81 0 CGG
82 0 TGAGAGG
83 0 CGC
84 0 GAGCA
85 0 GTGCTT
86 0 GTCC
87 0 CGCTG
88 0 CGCA
89 0 TCCCC
90 0 GCAGGTG
91 0

In [ ]:
attention = tuple(attn.detach().cpu() for attn in outputs.attentions)

model_view(
    attention,
    tokens,
    sentence_b_start=sentence_b_start,
)

In [ ]:
head_view(
    attention,
    tokens,
    sentence_b_start=sentence_b_start,
    include_layers=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
)